#### From Text to Diagnosis:
#### Evaluating GPT-Based Models for Classifying Depression Severity in Online Texts

# Script 4 - RAG

In [ ]:
# Instalation of dependencies.
pip install langchain langchain_community langchain_chroma langchain-openai

'''
As of the chroma upgrade to 0.5.4, chromadb needs to be depreciated for the notebooke to work,
due to a bug in the current version.
'''
pip install chromadb==0.5.3 --force-reinstall --user

In [ ]:
# Imports.

import getpass
import os
import pandas as pd

from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

In [ ]:
# Input OpenAI key through getpass() to ensure it remains hidden.

os.environ["OPENAI_API_KEY"] = getpass.getpass()

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

### DATA PREPARATION

In [ ]:
# Load the train set in a dataframe to prepare for load.
# The train set will be used for the example retrieval.
df = pd.read_csv(r"C:/Users/mike.w/Downloads/data/data-train.csv", delimiter=',')

# Selection of only the relevant columns.
df = df[['text', 'label_multi', 'domain']]
df = df.rename(columns={'label_multi': 'depression_level'})

# Save df to CSV.
df.to_csv(r"C:/Users/mike.w/Downloads/data/to_load.csv", index=False)

### LOADING

https://python.langchain.com/v0.2/docs/how_to/document_loader_csv/

https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/document_loaders/csv_loader.py#L98

https://api.python.langchain.com/en/latest/document_loaders/langchain_community.document_loaders.csv_loader.CSVLoader.html

In [ ]:
'''
Note that metadata is not included in the document, source however is.
This is why the depression_level is included as source,
to be ready for use by the model without further preprocessing.
'''

loader = CSVLoader(
    file_path = "C:/Users/theodore.balas/Downloads/data/to_load.csv",
    metadata_columns = ["domain"],
    source_column = "depression_level",
    encoding="utf-8"
)

data = loader.load()
for record in data:
    print(record)

### SPLITTING

In this itteration of RAG, splitting was not required, as the text was already in a format suitable for extraction (both due to small size, and high cohesion).

However, an investigation of the extraction of text smaller than the entirety of the post would be useful. Perhaps the text that infuences depression level the most lies in specific sentences within the examples. On the other hand, reducing the retrieved text size, increases the retrieval library, so filtering the most relevant examples could prove more challenging.

### EMBEDDING

https://api.python.langchain.com/en/latest/embeddings/langchain_core.embeddings.Embeddings.html

https://python.langchain.com/v0.2/docs/how_to/embed_text/

In [ ]:
vectorstore = Chroma.from_documents(documents=data, embedding=OpenAIEmbeddings())

### RETRIEVE

https://python.langchain.com/v0.2/docs/how_to/#retrievers

https://github.com/langchain-ai/langchain/discussions/9645

https://github.com/langchain-ai/langchain/issues/1838

In [ ]:
# Load the test set as a df.
test_set = pd.read_csv(r"C:/Users/theodore.balas/Downloads/data/data-test.csv", delimiter=',')

# Define the sources.
sources = ["minimum", "mild", "moderate", "severe"]

# Initialize cols in the DataFrame for storing results.
for source in sources:
    for num in range(1, 3):  # Two columns per source.
        test_set[f'{source}{num}'] = ""

# Iterate over each row in the df.
for index, row in test_set.iterrows():
    text = row['text']

    # Process each source and retrieve most and second most relevant example (k=2).
    for source in sources:
        BOOM = vectorstore.as_retriever(
            search_kwargs={'filter': {'source': source}, "k": 2}
        )
        result = BOOM.invoke(text)

        # If both results are available, store them in the df.
        if result and isinstance(result, list) and len(result) >= 2:
            # Extract page content from the document objects.
            test_set.at[index, f'{source}1'] = result[0].page_content
            test_set.at[index, f'{source}2'] = result[1].page_content
        
        # If only one result is retrieved, second col for this source is empty string.
        elif result and isinstance(result, list) and len(result) == 1:
            test_set.at[index, f'{source}1'] = result[0].page_content
            
        # If result is not a list or is empty, the both cols for this source are empty strings.

# After processing, view df.
print(test_set)

'''
Note that the error in the end of the code was due to a name conflict!
It runs properly!
'''

In [ ]:
# Check for missing vals.

missing_values_count = test_set.isna().sum()
print(missing_values_count)

In [ ]:
# Save the specified columns to a csv to be fed to the prompts.

columns_to_save = [
    'id', 'subreddit', 'post_id', 'sentence_range', 'text', 'label',
    'label_multi', 'domain', 'multi_label_number', 'strata',
    'minimum1', 'minimum2', 'mild1', 'mild2', 'moderate1', 'moderate2', 'severe1', 'severe2'
]

test_set.to_csv(
    r"C:/Users/theodore.balas/Downloads/data/processed_test_set_selected_columns.csv",
    columns=columns_to_save,
    index=False
)